In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.models import densenet121, maxvit_t
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def yolo_to_multilabel(label_path, num_classes=28):
    """ Convert YOLO labels to multi-label format (one-hot encoded). """
    labels = np.zeros(num_classes, dtype=np.float32)
    try:
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) > 0:
                    class_id = int(parts[0])
                    if 0 <= class_id < num_classes:
                        labels[class_id] = 1  
    except Exception as e:
        print(f"Error reading label file {label_path}: {e}")
    return labels

class WasteDataset(Dataset):
    def __init__(self, image_dir, label_dir, num_classes=28, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.num_classes = num_classes
        self.image_filenames = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]
    
    def __len__(self):
        return len(self.image_filenames)
    
    def __getitem__(self, idx):
        image_filename = self.image_filenames[idx]
        image_path = os.path.join(self.image_dir, image_filename)
        label_path = os.path.join(self.label_dir, image_filename.replace('.jpg', '.txt'))
        
        image = Image.open(image_path).convert("RGB")
        labels = yolo_to_multilabel(label_path, self.num_classes)
        
        if self.transform:
            image = self.transform(image)
        
        return image.to(device), torch.tensor(labels, dtype=torch.float32).to(device)

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Dataset and DataLoader
train_dataset = WasteDataset("C:/Users/Alfina/Downloads/archive/Warp-D/train/images", "C:/Users/Alfina/Downloads/archive/Warp-D/train/labels", transform=transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

test_dataset = WasteDataset("C:/Users/Alfina/Downloads/archive/Warp-D/test/images", "C:/Users/Alfina/Downloads/archive/Warp-D/test/labels", transform=transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Define 2S_DenseViT model
class TwoStreamDenseViT(nn.Module):
    def __init__(self, num_classes=28):
        super(TwoStreamDenseViT, self).__init__()
        self.densenet = densenet121(pretrained=True)
        self.densenet.classifier = nn.Identity()
        
        self.maxvit = maxvit_t(pretrained=True)
        self.maxvit.classifier = nn.Identity()
        
        self.pool = nn.AdaptiveAvgPool2d(1)
        
        self.fc = nn.Linear(1024 + 512, num_classes)
        
        # Freeze initial layers
        for param in list(self.densenet.features.parameters())[:50]:
            param.requires_grad = False
        for param in list(self.maxvit.parameters())[:50]:
            param.requires_grad = False
        
    def forward(self, x):
        densenet_features = self.densenet(x)
        maxvit_features = self.maxvit(x)
        maxvit_features = self.pool(maxvit_features)
        maxvit_features = torch.flatten(maxvit_features, start_dim=1)
        combined = torch.cat((densenet_features, maxvit_features), dim=1)
        return self.fc(combined)

# Initialize model
model = TwoStreamDenseViT(num_classes=28).to(device)

# Loss and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

def train_model(model, train_loader, criterion, optimizer, scheduler, num_epochs=10, early_stop_patience=3):
    model.train()
    best_loss = float('inf')
    patience = early_stop_patience
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")
        
        scheduler.step()
        
        # Early stopping
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience = early_stop_patience
            torch.save(model.state_dict(), r"C:\wasteproject\TwoDenseViT_model.pth")
            print("Model improved, saved as 2DenseViT_model.pth")
        else:
            patience -= 1
            if patience == 0:
                print("Early stopping triggered.")
                break

train_model(model, train_loader, criterion, optimizer, scheduler, num_epochs=10)

C:\Users\Alfina\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Alfina\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\Alfina\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchvision\models\_utils.py:22

Epoch 1/10, Loss: 0.3192
Model improved, saved as 2DenseViT_model.pth
Epoch 2/10, Loss: 0.2872
Model improved, saved as 2DenseViT_model.pth
Epoch 3/10, Loss: 0.2740
Model improved, saved as 2DenseViT_model.pth
Epoch 4/10, Loss: 0.2520
Model improved, saved as 2DenseViT_model.pth
Epoch 5/10, Loss: 0.2348
Model improved, saved as 2DenseViT_model.pth
Epoch 6/10, Loss: 0.1827
Model improved, saved as 2DenseViT_model.pth
Epoch 7/10, Loss: 0.1514
Model improved, saved as 2DenseViT_model.pth
Epoch 8/10, Loss: 0.1285
Model improved, saved as 2DenseViT_model.pth
Epoch 9/10, Loss: 0.1078
Model improved, saved as 2DenseViT_model.pth
Epoch 10/10, Loss: 0.0860
Model improved, saved as 2DenseViT_model.pth


In [13]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import numpy as np

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the image preprocessing function
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Load the trained model
model = TwoStreamDenseViT(num_classes=28).to(device)
model.load_state_dict(torch.load(r"C:\wasteproject\TwoDenseViT_model.pth"))
model.eval()

# Function to classify a single image
def classify_image(image_path, threshold=0.20):
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)  # Add batch dimension
    
    with torch.no_grad():
        output = torch.sigmoid(model(image))  # Get predictions
        predicted_labels = (output > threshold).cpu().numpy().astype(int)[0]  # Convert to binary labels
    
    # Load class names (assuming class names are stored in dataset.yaml)
    class_names = [f"Class {i}" for i in range(28)]  # Replace with actual class names if available
    
    # Get predicted class names
    detected_classes = [class_names[i] for i in range(28) if predicted_labels[i] == 1]
    
    return detected_classes

# Test with an image
image_path = r"C:\Users\Alfina\Desktop\major\Warp-D\test\images\Monitoring_photo_2_test_25-Mar_11-39-59.jpg"  # Replace with an actual image path
predictions = classify_image(image_path)  #22, 0

print("Predicted Classes:", predictions)


Predicted Classes: ['Class 0']


In [5]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_with_threshold(model, test_loader, thresholds):
    model.eval()
    best_f1 = 0
    best_threshold = 0
    results = {}

    with torch.no_grad():
        y_true = []
        y_pred_scores = []

        for images, labels in test_loader:
            outputs = torch.sigmoid(model(images))
            y_true.extend(labels.cpu().numpy())
            y_pred_scores.extend(outputs.cpu().numpy())

        y_true = np.array(y_true)
        y_pred_scores = np.array(y_pred_scores)

        for threshold in thresholds:
            y_pred = (y_pred_scores > threshold).astype(int)

            # ✅ Convert each row to a set (order doesn't matter)
            y_true_sets = [set(np.where(row == 1)[0]) for row in y_true]
            y_pred_sets = [set(np.where(row == 1)[0]) for row in y_pred]

            # ✅ Compute metrics ignoring order
            precision = np.mean([
                len(y_t & y_p) / len(y_p) if len(y_p) > 0 else 1
                for y_t, y_p in zip(y_true_sets, y_pred_sets)
            ]) * 100

            recall = np.mean([
                len(y_t & y_p) / len(y_t) if len(y_t) > 0 else 1
                for y_t, y_p in zip(y_true_sets, y_pred_sets)
            ]) * 100

            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

            results[threshold] = (precision, recall, f1)

            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold

    print("\n📊 Threshold Tuning Results (Order Ignored):")
    for t, (p, r, f) in results.items():
        print(f"  🔹 Threshold {t:.2f} → Precision: {p:.2f}%, Recall: {r:.2f}%, F1-score: {f:.2f}%")

    print(f"\n✅ Best Threshold: {best_threshold:.2f} with F1-score: {best_f1:.2f}%")
    return best_threshold

# Run evaluation
thresholds = np.linspace(0.1, 0.9, 9)
BEST_THRESHOLD = evaluate_with_threshold(model, test_loader, thresholds)



📊 Threshold Tuning Results (Order Ignored):
  🔹 Threshold 0.10 → Precision: 38.78%, Recall: 48.43%, F1-score: 43.07%
  🔹 Threshold 0.20 → Precision: 52.76%, Recall: 37.07%, F1-score: 43.54%
  🔹 Threshold 0.30 → Precision: 63.56%, Recall: 32.31%, F1-score: 42.84%
  🔹 Threshold 0.40 → Precision: 71.19%, Recall: 29.56%, F1-score: 41.77%
  🔹 Threshold 0.50 → Precision: 78.25%, Recall: 27.28%, F1-score: 40.46%
  🔹 Threshold 0.60 → Precision: 82.08%, Recall: 24.46%, F1-score: 37.69%
  🔹 Threshold 0.70 → Precision: 85.70%, Recall: 21.81%, F1-score: 34.77%
  🔹 Threshold 0.80 → Precision: 88.59%, Recall: 18.85%, F1-score: 31.08%
  🔹 Threshold 0.90 → Precision: 92.21%, Recall: 14.31%, F1-score: 24.77%

✅ Best Threshold: 0.20 with F1-score: 43.54%


In [15]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)  # Suppress warnings

import os
import torch
import numpy as np
import yaml
from sklearn.metrics import precision_score, recall_score, f1_score
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.models import densenet121, maxvit_t
import torch.nn as nn
# Custom dataset class
class WasteDataset(Dataset):
    def __init__(self, image_dir, label_dir, num_classes=28, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform
        self.num_classes = num_classes
        self.image_filenames = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]
    
    def __len__(self):
        return len(self.image_filenames)
    
    def __getitem__(self, idx):
        image_filename = self.image_filenames[idx]
        image_path = os.path.join(self.image_dir, image_filename)
        label_path = os.path.join(self.label_dir, image_filename.replace('.jpg', '.txt'))
        
        image = Image.open(image_path).convert("RGB")
        labels = yolo_to_multilabel(label_path, self.num_classes)
        
        if self.transform:
            image = self.transform(image)
        
        return image.to(device), torch.tensor(labels, dtype=torch.float32).to(device)

# Image transformations

# Define Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Define the Two-Stream DenseViT model
class TwoStreamDenseViT(nn.Module):
    def __init__(self, num_classes=28):
        super(TwoStreamDenseViT, self).__init__()
        self.densenet = densenet121(pretrained=True)
        self.densenet.classifier = nn.Identity()
        
        self.maxvit = maxvit_t(pretrained=True)
        self.maxvit.classifier = nn.Identity()
        
        self.pool = nn.AdaptiveAvgPool2d(1)
        
        self.fc = nn.Linear(1024 + 512, num_classes)
        
        # Freeze initial layers
        for param in list(self.densenet.features.parameters())[:50]:
            param.requires_grad = False
        for param in list(self.maxvit.parameters())[:50]:
            param.requires_grad = False
        
    def forward(self, x):
        densenet_features = self.densenet(x)
        maxvit_features = self.maxvit(x)
        maxvit_features = self.pool(maxvit_features)
        maxvit_features = torch.flatten(maxvit_features, start_dim=1)
        combined = torch.cat((densenet_features, maxvit_features), dim=1)
        return self.fc(combined)

# Load class names from dataset.yaml
yaml_path = r"C:/Users/Alfina/Downloads/archive/Warp-D/dataset.yaml"
with open(yaml_path, "r") as f:
    data_yaml = yaml.safe_load(f)
class_names = data_yaml["names"]

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

#Dataset and DataLoader
test_dataset = WasteDataset(
    "C:/Users/Alfina/Downloads/archive/Warp-D/test/images", 
    "C:/Users/Alfina/Downloads/archive/Warp-D/test/labels", 
    transform=transform
)#

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
# Load trained model
model = TwoStreamDenseViT(num_classes=len(class_names)).to(device)
model.load_state_dict(torch.load(r"C:\wasteproject\TwoDenseViT_model.pth", map_location=device))
model.eval()

def evaluate_model_accuracy_ignore_order(model, test_loader, best_threshold):
    model.eval()
    y_true = []
    y_pred_scores = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.cpu().numpy()  # Convert labels to numpy
            
            outputs = torch.sigmoid(model(images))
            y_true.extend(labels)
            y_pred_scores.extend(outputs.cpu().numpy())

    y_true = np.array(y_true)
    y_pred_scores = np.array(y_pred_scores)

    # Apply threshold
    y_pred = (y_pred_scores > best_threshold).astype(int)

    # Convert each row into a set of predicted and true class indices
    correct_predictions = []
    total_samples = y_true.shape[0]

    for true_labels, pred_labels in zip(y_true, y_pred):
        true_classes = set(np.where(true_labels == 1)[0])  # Set of ground truth labels
        pred_classes = set(np.where(pred_labels == 1)[0])  # Set of predicted labels

        # If all predicted classes exist in true classes, count as correct
        correct_predictions.append(len(pred_classes & true_classes) / (len(true_classes) + 1e-10))  

    # Compute accuracy ignoring order
    order_ignored_accuracy = np.mean(correct_predictions) * 100

    print(f"✅ Multi-Label Accuracy (Ignoring Order): {order_ignored_accuracy:.2f}%")

    return order_ignored_accuracy

# 🔹 Run evaluation
evaluate_model_accuracy_ignore_order(model, test_loader,BEST_THRESHOLD)


✅ Multi-Label Accuracy (Ignoring Order): 37.07%


np.float64(37.06881347511372)

In [19]:
from sklearn.metrics import f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
# ==== Evaluation Function ====
def evaluate_model(model, test_loader, threshold=0.2):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            preds = (probs > threshold).float()
            all_preds.append(preds.cpu().numpy())
            all_targets.append(labels.cpu().numpy())

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    micro_f1 = f1_score(all_targets, all_preds, average='micro')
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    precision = precision_score(all_targets, all_preds, average='micro')
    recall = recall_score(all_targets, all_preds, average='micro')

    print("\n--- Evaluation Metrics ---")
    print(f"Micro F1 Score: {micro_f1:.4f}")
    print(f"Macro F1 Score: {macro_f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")

    return all_preds, all_targets

# ==== Evaluate on Test Set ====
model.load_state_dict(torch.load(r"C:\wasteproject\TwoDenseViT_model.pth"))
evaluate_model(model, test_loader)


--- Evaluation Metrics ---
Micro F1 Score: 0.4694
Macro F1 Score: 0.4704
Precision: 0.4763
Recall: 0.4627


(array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 1.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], shape=(522, 28), dtype=float32),
 array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], shape=(522, 28), dtype=float32))

In [18]:
def predict_and_show(model, test_loader, threshold=0.2, num_images=50):
    model.eval()
    predictions = []
    actuals = []
    count = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            preds = (probs > threshold).float()

            for i in range(images.size(0)):
                if count >= num_images:
                    break
                pred_indices = torch.nonzero(preds[i]).squeeze().tolist()
                true_indices = torch.nonzero(labels[i]).squeeze().tolist()

                # Ensure we handle scalar tensor case
                if isinstance(pred_indices, int):
                    pred_indices = [pred_indices]
                if isinstance(true_indices, int):
                    true_indices = [true_indices]

                print(f"Image {count+1}")
                print(f"  Actual   : {sorted(true_indices)}")
                print(f"  Predicted: {sorted(pred_indices)}\n")

                count += 1
            if count >= num_images:
                break

# Load model and run prediction
model.load_state_dict(torch.load(r"C:\wasteproject\TwoDenseViT_model.pth"))
predict_and_show(model, test_loader, threshold=0.2, num_images=50)


Image 1
  Actual   : [11, 13, 14]
  Predicted: [11, 13, 14]

Image 2
  Actual   : [13]
  Predicted: [4, 13, 27]

Image 3
  Actual   : [4, 7]
  Predicted: [6]

Image 4
  Actual   : [6, 16]
  Predicted: [15]

Image 5
  Actual   : [3, 8, 13]
  Predicted: []

Image 6
  Actual   : [1, 2, 4]
  Predicted: [7]

Image 7
  Actual   : [8, 23]
  Predicted: [23]

Image 8
  Actual   : [0]
  Predicted: []

Image 9
  Actual   : [1, 10, 11]
  Predicted: [1, 6, 11]

Image 10
  Actual   : [0]
  Predicted: [17]

Image 11
  Actual   : [4]
  Predicted: [0, 6, 10]

Image 12
  Actual   : [1, 2]
  Predicted: [1, 2]

Image 13
  Actual   : [4]
  Predicted: []

Image 14
  Actual   : [10, 16]
  Predicted: [12, 16]

Image 15
  Actual   : [4, 9]
  Predicted: [9]

Image 16
  Actual   : [7]
  Predicted: [8]

Image 17
  Actual   : [2, 7, 16]
  Predicted: [2, 16]

Image 18
  Actual   : [10]
  Predicted: [6]

Image 19
  Actual   : [10]
  Predicted: [0, 3, 10]

Image 20
  Actual   : [4, 12]
  Predicted: [5]

Image 21
  Ac